In [1]:
import glob, os, sys, csv
import pandas as pd
from pyspark.sql import functions as F

In [2]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [16]:
query = \
  """SELECT TIPOBITO
          ,DTOBITO
          ,HORAOBITO
          ,NATURAL
          ,CODMUNNATU
          ,DTNASC
          ,IDADE
          ,SEXO
          ,RACACOR
          ,ESTCIV
          ,ESC2010
          ,SERIESCFAL
          ,OCUP
          ,CODMUNRES
          ,LOCOCOR
          ,CODESTAB
          ,CODMUNOCOR
          ,IDADEMAE
          ,ESCMAE2010
          ,SERIESCMAE
          ,OCUPMAE
          ,QTDFILVIVO
          ,QTDFILMORT
          ,GRAVIDEZ
          ,SEMAGESTAC
          ,PARTO
          ,OBITOPARTO
          ,PESO
          ,TPMORTEOCO
          ,ASSISTMED
          ,NECROPSIA
          ,LINHAA
          ,LINHAB
          ,LINHAC
          ,LINHAD
          ,LINHAII
          ,CAUSABAS
          ,ATESTANTE
          ,COMUNSVOIM
          ,DTATESTADO
          ,CIRCOBITO
          ,ACIDTRAB
          ,FONTE
          ,TPOBITOCOR
          ,ORIGEM
          ,ESC
          ,ESCMAE
          ,OBITOGRAV
          ,OBITOPUERP
          ,EXAME
          ,CIRURGIA
          ,CAUSABAS_O
          ,NUMEROLOTE
          ,DTINVESTIG
          ,DTCADASTRO
          ,STCODIFICA
          ,CODIFICADO
          ,VERSAOSIST
          ,VERSAOSCB
          ,FONTEINV
          ,DTRECEBIM
          ,ATESTADO
          ,DTRECORIGA
          ,OPOR_DO
          ,CAUSAMAT
          ,ESCMAEAGR1
          ,ESCFALAGR1
          ,STDOEPIDEM
          ,STDONOVA
          ,DIFDATA
          ,NUDIASOBCO
          ,DTCADINV
          ,DTCONINV
          ,FONTES
          ,TPRESGINFO
          ,TPNIVELINV
          ,DTCADINF
          ,MORTEPARTO
          ,DTCONCASO
          ,ALTCAUSA
          ,TPPOS
          ,TP_ALTERA
          ,GESTACAO
          ,CB_ALT
      FROM mortalidade_temp"""

In [22]:
pasta_arquivos = r"C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv"  # Altere para o caminho da sua pasta, se necessário

padrao_busca = os.path.join(pasta_arquivos, "Mortalidade_Geral_*.csv")
arquivos = sorted(glob.glob(padrao_busca))

for i, arquivo in enumerate(arquivos):
    print(f"Processando arquivo: {arquivo}", end="")

    df_ = spark.read.csv(arquivo, header=True, inferSchema=True, sep=';')

    ls_df_cols = df_.columns

    # print(ls_df_cols)

    if "OPOR_DO" not in ls_df_cols:
        df_ = df_.withColumn("OPOR_DO", F.lit(""))

    if "TP_ALTERA" not in ls_df_cols:
        df_ = df_.withColumn("TP_ALTERA", F.lit(""))

    if "CB_ALT" not in ls_df_cols:
        df_ = df_.withColumn("CB_ALT", F.lit(""))

    if "GESTACAO" not in ls_df_cols:
        df_ = df_.withColumn("GESTACAO", F.lit(""))

    if "ALTCAUSA" not in ls_df_cols:
        df_ = df_.withColumn("ALTCAUSA", F.lit(""))

    if "CAUSABAS_O" not in ls_df_cols:
        df_ = df_.withColumn("CAUSABAS_O", F.lit(""))

    if "TPPOS" not in ls_df_cols:
        df_ = df_.withColumn("TPPOS", F.lit(""))

    df_.createOrReplaceTempView("mortalidade_temp")
    
    df_cols = spark.sql(query)
    # Adiciona aS colunaS NOME_ARQ e ANO com base no nome do arquivo
    df_cols = \
        (df_cols.withColumns({"ANO": F.regexp_extract(F.lit(arquivo) , r"_(\d{4})(?:_|\.csv)", 1)
                             ,"NOME_ARQ": F.lit(arquivo)
                             }))

    df_cols.toPandas().to_csv(f"{pasta_arquivos}\\normalizar_colunas\\{os.path.basename(arquivo)}"
                             ,sep=';',encoding='utf-8'
                             ,quoting=csv.QUOTE_ALL
                             ,index=False)

    print(" - csv criado com sucesso ", end = "")

    if i == 0:
        df_final = df_cols
    else:
        df_final = df_final.unionByName(df_cols)

    print(" - Union Ok ")

    # if i >= 2:
    #     break

# df_cols.printSchema()

Processando arquivo: C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2014.csv - csv criado com sucesso  - Union Ok 
Processando arquivo: C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2015.csv - csv criado com sucesso  - Union Ok 
Processando arquivo: C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2016.csv - csv criado com sucesso  - Union Ok 
Processando arquivo: C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2017.csv - csv criado com sucesso  - Union Ok 
Processando arquivo: C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2018.csv - csv criado com sucesso  - Union Ok 
Processando arquivo: C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2019.csv - csv criado com sucesso  - Union Ok 
Processando arquivo: C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2020_part01.csv - csv criado com sucesso  - Union Ok 
Processando a

In [19]:
df_final.printSchema()
# if "OPOR_DO" not in ls_df_cols:
#     print("OK")
# else:
#     print("NOK")

root
 |-- TIPOBITO: integer (nullable = true)
 |-- DTOBITO: integer (nullable = true)
 |-- HORAOBITO: long (nullable = true)
 |-- NATURAL: integer (nullable = true)
 |-- CODMUNNATU: integer (nullable = true)
 |-- DTNASC: integer (nullable = true)
 |-- IDADE: integer (nullable = true)
 |-- SEXO: integer (nullable = true)
 |-- RACACOR: integer (nullable = true)
 |-- ESTCIV: integer (nullable = true)
 |-- ESC2010: integer (nullable = true)
 |-- SERIESCFAL: integer (nullable = true)
 |-- OCUP: integer (nullable = true)
 |-- CODMUNRES: integer (nullable = true)
 |-- LOCOCOR: integer (nullable = true)
 |-- CODESTAB: integer (nullable = true)
 |-- CODMUNOCOR: integer (nullable = true)
 |-- IDADEMAE: integer (nullable = true)
 |-- ESCMAE2010: integer (nullable = true)
 |-- SERIESCMAE: integer (nullable = true)
 |-- OCUPMAE: integer (nullable = true)
 |-- QTDFILVIVO: integer (nullable = true)
 |-- QTDFILMORT: integer (nullable = true)
 |-- GRAVIDEZ: integer (nullable = true)
 |-- SEMAGESTAC: in

In [21]:
df_final.select("NOME_ARQ", "ANO").groupBy("NOME_ARQ", "ANO").count().limit(1000).show(truncate = False)
# filter("ANO >= '2020'").

+--------------------------------------------------------------------------------+----+-------+
|NOME_ARQ                                                                        |ANO |count  |
+--------------------------------------------------------------------------------+----+-------+
|C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2014.csv|2014|1227039|
|C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2015.csv|2015|1264175|
|C:\Marco Conti\Projetos\Dados\DataSUS\Mortalidade\csv\Mortalidade_Geral_2016.csv|2016|1309774|
+--------------------------------------------------------------------------------+----+-------+

